In [1]:
import pandas as pd
import sklearn.preprocessing

df = pd.read_csv('adult.csv')
print(df.head())

   age workclass  fnlwgt     education  education.num marital.status  \
0   90         ?   77053       HS-grad              9        Widowed   
1   82   Private  132870       HS-grad              9        Widowed   
2   66         ?  186061  Some-college             10        Widowed   
3   54   Private  140359       7th-8th              4       Divorced   
4   41   Private  264663  Some-college             10      Separated   

          occupation   relationship   race     sex  capital.gain  \
0                  ?  Not-in-family  White  Female             0   
1    Exec-managerial  Not-in-family  White  Female             0   
2                  ?      Unmarried  Black  Female             0   
3  Machine-op-inspct      Unmarried  White  Female             0   
4     Prof-specialty      Own-child  White  Female             0   

   capital.loss  hours.per.week native.country income  
0          4356              40  United-States  <=50K  
1          4356              18  United-States

In [2]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

df['workclass_encoded'] = le.fit_transform(df['workclass'])

print("Workclass labels mapping:", dict(zip(le.classes_, le.transform(le.classes_))))
print(df[['workclass', 'workclass_encoded']].head())

Workclass labels mapping: {'?': np.int64(0), 'Federal-gov': np.int64(1), 'Local-gov': np.int64(2), 'Never-worked': np.int64(3), 'Private': np.int64(4), 'Self-emp-inc': np.int64(5), 'Self-emp-not-inc': np.int64(6), 'State-gov': np.int64(7), 'Without-pay': np.int64(8)}
  workclass  workclass_encoded
0         ?                  0
1   Private                  4
2         ?                  0
3   Private                  4
4   Private                  4


In [3]:
from sklearn.preprocessing import OneHotEncoder

categorical_cols = ['education', 'relationship', 
                    'race', 'sex', 'marital.status']
ohe = OneHotEncoder(sparse_output=False)
ohe_array = ohe.fit_transform(df[categorical_cols])
print("OHE features names:", ohe.get_feature_names_out(categorical_cols))

ohe_df = pd.DataFrame(
    ohe_array, columns=ohe.get_feature_names_out(categorical_cols))
df_ohe = pd.concat([df.reset_index(drop=True), ohe_df], axis=1)
print(df_ohe.head())

OHE features names: ['education_10th' 'education_11th' 'education_12th' 'education_1st-4th'
 'education_5th-6th' 'education_7th-8th' 'education_9th'
 'education_Assoc-acdm' 'education_Assoc-voc' 'education_Bachelors'
 'education_Doctorate' 'education_HS-grad' 'education_Masters'
 'education_Preschool' 'education_Prof-school' 'education_Some-college'
 'relationship_Husband' 'relationship_Not-in-family'
 'relationship_Other-relative' 'relationship_Own-child'
 'relationship_Unmarried' 'relationship_Wife' 'race_Amer-Indian-Eskimo'
 'race_Asian-Pac-Islander' 'race_Black' 'race_Other' 'race_White'
 'sex_Female' 'sex_Male' 'marital.status_Divorced'
 'marital.status_Married-AF-spouse' 'marital.status_Married-civ-spouse'
 'marital.status_Married-spouse-absent' 'marital.status_Never-married'
 'marital.status_Separated' 'marital.status_Widowed']
   age workclass  fnlwgt     education  education.num marital.status  \
0   90         ?   77053       HS-grad              9        Widowed   
1   82   

In [4]:
from sklearn.preprocessing import OrdinalEncoder

# Step 1: Define ordinal column
ordinal_cols = ['education']

# Step 2: Define category order (LOW → HIGH education)
categories_order = [[
    'Preschool',
    '1st-4th',
    '5th-6th',
    '7th-8th',
    '9th',
    '10th',
    '11th',
    '12th',
    'HS-grad',
    'Some-college',
    'Assoc-voc',
    'Assoc-acdm',
    'Bachelors',
    'Masters',
    'Prof-school',
    'Doctorate'
]]

# Step 3: Apply OrdinalEncoder
oe = OrdinalEncoder(categories=categories_order)

df['education_ord'] = oe.fit_transform(df[ordinal_cols])

# Step 4: Show result
print(df[['education', 'education_ord']].head())


      education  education_ord
0       HS-grad            8.0
1       HS-grad            8.0
2  Some-college            9.0
3       7th-8th            3.0
4  Some-college            9.0


In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder

# Ordinal feature
ordinal_features = ['education']

education_order = [[
    'Preschool',
    '1st-4th',
    '5th-6th',
    '7th-8th',
    '9th',
    '10th',
    '11th',
    '12th',
    'HS-grad',
    'Some-college',
    'Assoc-voc',
    'Assoc-acdm',
    'Bachelors',
    'Masters',
    'Prof-school',
    'Doctorate'
]]

# Nominal features (no natural order)
nominal_features = [
    'workclass',
    'marital.status',
    'occupation',
    'relationship',
    'race',
    'sex',
    'native.country'
]

# ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('ord', OrdinalEncoder(categories=education_order), ordinal_features),
        ('nom', OneHotEncoder(sparse_output=False, handle_unknown='ignore'), nominal_features)
    ]
)

# Combine selected features
features = ordinal_features + nominal_features
X = df[features]

# Transform
X_prepared = preprocessor.fit_transform(X)

print("Transformed shape:", X_prepared.shape)

Transformed shape: (32561, 87)


In [6]:
import numpy as np

final_df = pd.DataFrame(
    np.hstack([X_prepared, df[['workclass_encoded']].values]),
    columns = list(preprocessor.get_feature_names_out()) +
['workclass_encoded']
)
print(final_df.head())

   ord__education  nom__workclass_?  nom__workclass_Federal-gov  \
0             8.0               1.0                         0.0   
1             8.0               0.0                         0.0   
2             9.0               1.0                         0.0   
3             3.0               0.0                         0.0   
4             9.0               0.0                         0.0   

   nom__workclass_Local-gov  nom__workclass_Never-worked  \
0                       0.0                          0.0   
1                       0.0                          0.0   
2                       0.0                          0.0   
3                       0.0                          0.0   
4                       0.0                          0.0   

   nom__workclass_Private  nom__workclass_Self-emp-inc  \
0                     0.0                          0.0   
1                     1.0                          0.0   
2                     0.0                          0.0   
3   